# Validate the local floor-plan API

Start the server with `LAYOUT_BUNDLE_DIR` pointing at the exported bundle. Run this notebook with the same variable set. `MODEL_URL` defaults to the local server. A collection example can return itself and serves as an integration check.

In [ ]:
import base64
import io
import json
import os
from pathlib import Path

import numpy as np
import httpx
from gradio_client import Client
from IPython.display import display
from PIL import Image

here = Path.cwd()
if not (here / 'examples').is_dir():
    here = here / '02_model_server'
example = here / 'examples/plan_high_quality_architectural_1191.png'
model_url = os.environ.get('MODEL_URL', 'http://127.0.0.1:7860')
bundle = Path(os.environ.get('LAYOUT_BUNDLE_DIR', here.parent / '01_autoencoder/data/layout_aligned_v4/bundle'))
image_b64 = base64.b64encode(example.read_bytes()).decode('ascii')

client = Client(model_url, verbose=False)
prediction = client.predict(image_b64, api_name='/predict')
embedding = client.predict(image_b64, api_name='/embed')


## Verify both endpoints

Compare the returned IDs, distances, atlas coordinates and annotation flags against the exported collection. Recompute distances from the returned embedding, allowing for the API?s rounding.

In [ ]:
version = json.loads((bundle / 'config.json').read_text(encoding='utf-8'))['version']
assert prediction['model_version'] == embedding['model_version'] == version
with np.load(bundle / 'collection.npz', allow_pickle=False) as collection:
    ids, codes, atlas, flags = (collection[key] for key in ('ids', 'codes', 'atlas', 'has_rooms'))

code = np.asarray(embedding['code'], dtype=np.float64)
assert embedding['size'] == 128 and code.shape == (128,) and np.isfinite(code).all()
matches = prediction['matches']
assert len(matches) == 5 and [m['rank'] for m in matches] == list(range(1, 6))
id_to_row = {plan_id: i for i, plan_id in enumerate(ids)}
rows = [id_to_row[match['id']] for match in matches]
assert len(set(rows)) == 5
distances = np.linalg.norm(codes - code, axis=1)
returned = np.array([match['distance'] for match in matches])
assert np.isfinite(returned).all() and np.all(np.diff(returned) >= 0)
np.testing.assert_allclose(returned, distances[rows], rtol=0, atol=.0051)
np.testing.assert_allclose(np.sort(distances[rows]), np.sort(distances)[:5], rtol=0, atol=.0002)
np.testing.assert_allclose(prediction['atlas_xy'], atlas[rows].mean(axis=0), rtol=0, atol=.000051)
for match, row in zip(matches, rows):
    np.testing.assert_allclose(match['atlas_xy'], atlas[row], rtol=0, atol=.000051)
    assert match['has_rooms'] == bool(flags[row])
    assert bool(match['rooms_png_base64']) == bool(flags[row])
    print(match['rank'], match['id'], match['distance'])


## Verify the plain HTTP integration

In [ ]:
def http_call(name):
    endpoint = f'{model_url}/gradio_api/call/{name}'
    ticket = httpx.post(endpoint, json={'data': [image_b64]}, timeout=60)
    ticket.raise_for_status()
    answer = httpx.get(f"{endpoint}/{ticket.json()['event_id']}", timeout=60)
    answer.raise_for_status()
    for event in answer.text.split('\n\n'):
        lines = event.splitlines()
        if lines and lines[0] == 'event: complete':
            return json.loads(next(line[6:] for line in lines if line.startswith('data: ')))[0]
    raise RuntimeError(f'No completed event for {name}: {answer.text[:200]}')

assert http_call('predict') == prediction
assert http_call('embed') == embedding


## Display query predictions and gallery annotations

In [ ]:
def from_base64(text):
    with Image.open(io.BytesIO(base64.b64decode(text, validate=True))) as image:
        assert image.format == 'PNG' and image.size == (128, 128)
        return image.convert('RGB')

query = Image.open(example).convert('RGB')
query.thumbnail((128, 128))
sheet = Image.new('RGB', (6 * 136, 2 * 148), 'white')
from PIL import ImageDraw
labels = ImageDraw.Draw(sheet)
for col, (plan, label, room_data) in enumerate([
    (query, 'Query prediction', prediction['rooms_png_base64']),
    *((from_base64(m['image_png_base64']), f"Match {m['rank']} annotation", m['rooms_png_base64'])
      for m in matches),
]):
    sheet.paste(plan, (col * 136, 0))
    labels.text((col * 136, 130), label, fill='black')
    if room_data:
        sheet.paste(from_base64(room_data), (col * 136, 148))
    else:
        labels.text((col * 136, 148), 'Unavailable', fill='black')
display(sheet)
